# `EukaryoticToeholdGate` — usage example

A minimal, real, end-to-end run of the single-input eukaryotic toehold switch —
`EukaryoticToeholdGate` in `engine.gates.toehold`. This class is fully implemented
and tested (`tests/engine/gates/test_toehold.py`, 29 passing tests as of this
writing). This is a sibling to [`toehold.ipynb`](toehold.ipynb) (which drives the
gate through a stub `FoldEngine` for fast, dependency-free iteration and covers both
hosts generically) — here we build `EukaryoticToeholdGate` specifically and use the
**real** `FoldEngine` (ViennaRNA) throughout, so every number below is a genuine
fold, not a placeholder.

No Django, no worker, no pipeline — just the gate class, constructed and called
directly, the way `pipeline.py` would use it internally.

## Setup

In [ ]:
# Put <repo>/src on the path. Search upward from cwd for pyproject.toml so this works
# wherever Jupyter is launched from.
import sys
from pathlib import Path

for _base in (Path.cwd(), *Path.cwd().parents):
    if (_base / "pyproject.toml").exists():
        _src = _base / "src"
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

from engine.domain import Host, Regulation, SelectedGene, TriggerSet, Constraints
from engine.gates.toehold import EukaryoticToeholdGate
from engine.gates.tools.folding import FoldEngine
from engine.gates.tools.translation import TranslationScorer
from engine.gates.tools.codons import CodonOptimizer
from engine.stages.folding import FoldProfiler
from engine.stages.motifs import MotifScreener
from engine.stages.off_target import OffTargetScanner
from engine.stages.triggers import TriggerScorer
from engine import sequences as sq

## 1. Build the tools, once

Per `CLAUDE.md` §5: tools are constructed once and handed to the gate, never built
inside a stage or family. `FoldEngine`'s cache is only useful if every caller shares
one instance — a second `FoldEngine()` means a cold cache and, worse, a second
chance to fold at a different temperature.

In [ ]:
host = Host.HUMAN  # HUMAN | YEAST both take the eukaryotic (Kozak) track

folder = FoldEngine(temperature=37.0)
translation = TranslationScorer(host)
codons = CodonOptimizer(host)

gate = EukaryoticToeholdGate(host, folder, translation, codons)
print(gate.required_tools())

## 2. Pick a trigger — via the real `TriggerScorer` (stage 2)

`TriggerScorer.score` is fully implemented (`tests/engine/test_triggers.py`, 47
passing tests): it slides every window of every length in
`constraints.trigger_lengths` across the transcript, screens out forbidden motifs,
folds each survivor for `openness`/`accessibility`/`mfe` via the same `FoldEngine`
the gate uses, checks off-targets, ranks by `accessibility * segment_specificity`,
and yields the top candidates per gene — so we use it for real here instead of
hand-picking one window.

`OffTargetScanner` itself is still a stub (`find_similar`/`scan_trigger` raise
`NotImplementedError`) — **except** when handed an empty transcriptome, which is a
deliberate early-return for exactly this case (a `direct` submission, or a demo like
this one, with no reference index to scan against): `scan_trigger` returns a clean
`OffTargetReport(hits=(), penalty=0.0)` rather than raising. That is a real,
documented behaviour of the class, not a workaround.

In [ ]:
# The full-length human AREG mRNA, as cDNA/DNA notation (T, not U) — same alphabet trap
# CLAUDE.md warns about, so normalise with to_rna() before anything else touches it.
transcript_dna = (
    "AGACGTTCGCACACCTGGGTGCCAGCGCCCCAGAGGTCCCGGGACAGCCCGAGGCGCCGCGCCCGCCGCCCCGAGCTCCCC"
    "AAGCCTTCGAGAGCGGCGCACACTCCCGGTCTCCACTCGCTCTTCCAACACCCGCTCGTTTTGGCGGCAGCTCGTGTCCCA"
    "GAGACCGAGTTGCCCCAGAGACCGAGACGCCGCCGCTGCGAAGGACCAATGAGAGCCCCGCTGCTACCGCCGGCGCCGGTG"
    "GTGCTGTCGCTCTTGATACTCGGCTCAGGCCATTATGCTGCTGGATTGGACCTCAATGACACCTACTCTGGGAAGCGTGAA"
    "CCATTTTCTGGGGACCACAGTGCTGATGGATTTGAGGTTACCTCAAGAAGTGAGATGTCTTCAGGGAGTGAGATTTCCCCT"
    "GTGAGTGAAATGCCTTCTAGTAGTGAACCGTCCTCGGGAGCCGACTATGACTACTCAGAAGAGTATGATAACGAACCACAA"
    "ATACCTGGCTATATTGTCGATGATTCAGTCAGAGTTGAACAGGTAGTTAAGCCCCCCCAAAACAAGACGGAAAGTGAAAAT"
    "ACTTCAGATAAACCCAAAAGAAAGAAAAAGGGAGGCAAAAATGGAAAAAATAGAAGAAACAGAAAGAAGAAAAATCCATGT"
    "AATGCAGAATTTCAAAATTTCTGCATTCACGGAGAATGCAAATATATAGAGCACCTGGAAGCAGTAACATGCAAATGTCA"
    "GCAAGAATATTTCGGTGAACGGTGTGGGGAAAAGTCCATGAAAACTCACAGCATGATTGACAGTAGTTTATCAAAAATTG"
    "CATTAGCAGCCATAGCTGCCTTTATGTCTGCTGTGATCCTCACAGCTGTTGCTGTTATTACAGTCCAGCTTAGAAGACAA"
    "TACGTCAGGAAATATGAAGGAGAAGCTGAGGAACGAAAGAAACTTCGACAAGAGAATGGAAATGTACATGCTATAGCATA"
    "ACTGAAGATAAAATTACAGGATATCACATTGGAGTCACTGCCAAGTCATAGCCATAAATGATGAGTCGGTCCTCTTTCCA"
    "GTGGATCATAAGACAATGGACCCTTTTTGTTATGATGGTTTTAAACTTTCAATTGTCACTTTTTATGCTATTTCTGTATA"
    "TAAAGGTGCACGAAGGTAAAAAGTATTTTTTCAAGTTGTAAATAATTTATTTAATATTTAATGGAAGTGTATTTATTTTA"
    "CAGCTCATTAAACTTTTTTAACCAAA"
)
transcript = sq.to_rna(transcript_dna)
assert sq.is_valid_rna(transcript)
print(f"transcript length: {len(transcript)} nt")

In [ ]:
# Stage-2 tools, built once (same injection rule as the gate's own tools).
profiler = FoldProfiler()
screener = MotifScreener()
off_target = OffTargetScanner(transcriptome={})  # empty: no reference index for this demo
scorer = TriggerScorer(profiler, off_target, screener, folder)  # shares the gate's FoldEngine

# In a real run this comes from GeneSelector (stage 1); stand in with a minimal
# SelectedGene since this demo starts from a single known transcript.
gene = SelectedGene(
    gene_id="AREG",
    symbol="AREG",
    regulation=Regulation.UP,
    log2_fold_change=2.0,
    score=1.0,
)
constraints = Constraints(trigger_lengths=(30, 36), max_switch_length=200)

candidates = list(scorer.score([gene], {"AREG": transcript}, constraints))
print(f"{len(candidates)} candidate(s), best-scoring first\n")
for c in candidates[:5]:
    print(
        f"  {c.trigger_id:22} start={c.start_index:4} len={c.length:2}  "
        f"score={c.score:.3f}  accessibility={c.accessibility:.3f}  gc={c.gc_content:.1f}"
    )

trigger = candidates[0]
print("\nselected:", trigger)

## 3. Wrap the trigger in a `TriggerSet`

`TriggerSet` is the circuit's inputs (one activator here — a single-input switch).
`Constraints` were already built above, since `TriggerScorer` needed them too — a run
builds `Constraints` once from `params["constraints"]` and threads the same object
through every stage.

In [ ]:
triggers = TriggerSet(activators=(trigger,))

print("arity:", triggers.arity, "| logic:", triggers.logic_type)

## 4. `is_compatible()` — cheap check before generating anything

Arity, host, trigger length window — nothing here folds.

In [ ]:
compatibility = gate.is_compatible(triggers, constraints)
print(compatibility)
assert compatibility.ok, compatibility.reason

## 5. `generate_designs()` — candidate switches

A generator — one `GateDesign` per architecture variant it yields (here, per
sweepable toehold length in `constraints.trigger_lengths` that the trigger can
support). Materialise with `list(...)` for a handful of designs; a real run would
consume this lazily.

In [ ]:
designs = list(gate.generate_designs(triggers, constraints))
print(f"{len(designs)} design(s)")
for d in designs:
    print(f"  {d.design_id}  {d.length} nt")

## 6. `evaluate_design()` — raw metrics for one design

**Raw** values only — no normalising, weighting or ranking here, that is
`engine.scoring`'s job. Keys are exactly the metric names `DEFAULT_V1` declares.

In [ ]:
design = designs[0]
metrics = gate.evaluate_design(design)
for name, value in metrics.items():
    print(f"{name:22} {value}")

Unlike an arbitrarily-chosen window, this trigger was `TriggerScorer`'s top pick by
accessibility — so `predicted_leakage` should come in well under the 0.85 hard-filter
threshold and `trigger_accessibility` should be well above the near-zero values a
poorly-chosen window produces. Run the cells above with a different `trigger_lengths`
tuple, or a different transcript, to see how the ranking responds.

## 7. `emit_sequence()` — the synthesis-ready sequence

In [ ]:
print(gate.emit_sequence(design))

The Kozak element and start codon aren't visually obvious in that raw string — they're
embedded in the middle of the construct, not set apart. Locate them explicitly using
`design.architecture` (which records `aug_index` exactly) and the gate's own
`KOZAK_EUKARYOTIC` constant:

In [ ]:
seq = design.sequence
aug_index = design.architecture["aug_index"]
kozak_index = seq.find(gate.KOZAK_EUKARYOTIC)
assert kozak_index != -1, "Kozak element not found — architecture assumptions above are stale"

marks = [" "] * len(seq)
for i in range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)):
    marks[i] = "K"
for i in range(aug_index, aug_index + 3):
    marks[i] = "A"

print(seq)
print("".join(marks), " K = Kozak (GCCACC)   A = start codon (AUG)")

## Where this fits in a real run

This notebook stops at one gate, one design. In the actual pipeline:

- `generate_designs` is called for **every** trigger set that passed `is_compatible`,
  yielding potentially thousands of designs.
- Every design's raw metrics go through `engine.scoring` — `build_metrics`,
  `weighted_score`, `failed_filter`, `rank_candidates` — which is what actually
  decides which designs survive and how they rank, comparably with every other gate
  family's designs.
- `CandidateStore` records provenance and writes the stage snapshot; nothing here
  hand-rolls a results CSV.

The two-input AND version (`EukaryoticToeholdAndGate`) is **not** implemented yet —
its `generate_designs` still raises `NotImplementedError("Step 5")`. This notebook
only covers the single-input case.